# Load groundwork 

In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


import scanpy as sc

import os

import scvi

import seaborn as sns

%matplotlib inline


/home/syyang/python_virtuenv/mrvi_3.11/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/syyang/python_virtuenv/mrvi_3.11/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/home/syyang/python_virtuenv/mrvi_3.11/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/home/syyang/python_virtuenv/mrvi_3.11/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing CSCDataset from `anndata.experimental` is deprecated. Import anndata.abc.CSCDataset instead.
  warnings.warn(msg, FutureWarning)
/home/syyang/python_virtuenv/mrvi_3.11/lib/python3.11/site-packages/anndat

In [2]:
## Downloaded CR-arc count matrix results
data_dir = '/mnt/hdd_bob/syy/adipose/atac/protocol_benchmark/cr_results/rna/VIB_10xmultiome_2_rna'

h5_file = os.path.join(data_dir, 'outs/raw_feature_bc_matrix.h5')


h5 = sc.read_10x_h5(h5_file) 

/home/syyang/python_virtuenv/mrvi_3.11/lib/python3.11/site-packages/anndata/_core/anndata.py:1760: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/syyang/python_virtuenv/mrvi_3.11/lib/python3.11/site-packages/anndata/_core/anndata.py:1760: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [3]:
h5

AnnData object with n_obs × n_vars = 666640 × 36601
    var: 'gene_ids', 'feature_types', 'genome'

In [4]:
h5.obs.head(2)

""
AAACAGCCAAACAACA-1
AAACAGCCAAACATAG-1


In [5]:
h5.var.head(3), 

(                    gene_ids    feature_types  genome
 MIR1302-2HG  ENSG00000243485  Gene Expression  GRCh38
 FAM138A      ENSG00000237613  Gene Expression  GRCh38
 OR4F5        ENSG00000186092  Gene Expression  GRCh38,)

## Matching atac bc with RNA bc

In [6]:
# atac-rna-barcode map 
this_dir = '/home/syyang/GitRepo/atac/EDA'

bc_map_file = f'{this_dir}/Info_from_CellRanger/atac_rna_barcodes_map.tsv'
bc_map_pd = pd.read_csv(bc_map_file, sep='\t')

In [7]:
bc_map_pd.head()

,atac_barcodes,rna_barcodes
0,ACAGCGGGTGTGTTAC,AAACAGCCAAACAACA
1,ACAGCGGGTTGTTCTT,AAACAGCCAAACATAG
2,ACAGCGGGTAACAGGC,AAACAGCCAAACCCTA
3,ACAGCGGGTGCGCGAA,AAACAGCCAAACCTAT
4,ACAGCGGGTCCTCCAT,AAACAGCCAAACCTTG


# RNA ad

In [8]:
h5_rna_ = h5.copy()
h5_rna_.obs['rna_barcodes'] = h5_rna_.obs.index.map(lambda x: x.split('-1')[0])
h5_rna_.obs = h5_rna_.obs.merge(bc_map_pd.set_index('rna_barcodes'), left_on='rna_barcodes', right_index=True, how='inner')

# calculate total RNA QC metrics
h5_rna_.obs['total_RNA_UMI'] = h5_rna_.X.sum(axis=1)
h5_rna_.obs['total_RNA_UMI_log'] = np.log1p(h5_rna_.obs['total_RNA_UMI'])
h5_rna_.obs['n_genes_by_RNA_counts'] = (h5_rna_.X > 0).sum(axis=1)
h5_rna_.obs['N_MT_counts'] = h5_rna_.X[:, h5_rna_.var.index.str.startswith('MT-')].sum(axis=1)
h5_rna_.obs['MT%'] = h5_rna_.obs['N_MT_counts'] / h5_rna_.obs['total_RNA_UMI'] * 100


/home/syyang/python_virtuenv/mrvi_3.11/lib/python3.11/site-packages/anndata/_core/anndata.py:1760: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [9]:
h5_rna_

AnnData object with n_obs × n_vars = 666640 × 36601
    obs: 'rna_barcodes', 'atac_barcodes', 'total_RNA_UMI', 'total_RNA_UMI_log', 'n_genes_by_RNA_counts', 'N_MT_counts', 'MT%'
    var: 'gene_ids', 'feature_types', 'genome'

In [10]:
h5_rna_.obs.head(2)

,rna_barcodes,atac_barcodes,total_RNA_UMI,total_RNA_UMI_log,n_genes_by_RNA_counts,N_MT_counts,MT%
AAACAGCCAAACAACA-1,AAACAGCCAAACAACA,ACAGCGGGTGTGTTAC,2.0,1.098612,2,1.0,50.0
AAACAGCCAAACATAG-1,AAACAGCCAAACATAG,ACAGCGGGTTGTTCTT,2.0,1.098612,2,1.0,50.0


In [11]:
h5_rna_.var.head(2)

,gene_ids,feature_types,genome
MIR1302-2HG,ENSG00000243485,Gene Expression,GRCh38
FAM138A,ENSG00000237613,Gene Expression,GRCh38


In [13]:
entropy_dir = '/mnt/hdd_bob/syy/adipose/atac/res/VIB_10xmultiome_2_WS3000F'
RNA_info_dir = os.path.join(entropy_dir, '_RNA_info')
if not os.path.exists(RNA_info_dir):
    os.makedirs(RNA_info_dir)
h5_rna_.write_h5ad(os.path.join(RNA_info_dir, 'allbc_rna.h5ad'))

In [16]:
union_bc_df

NameError: name 'union_bc_df' is not defined

In [17]:
union_bc_file = os.path.join(entropy_dir, '_cell_calling_comparison/Union_cell_3set.tsv')
union_bc_df = pd.read_csv(union_bc_file,  sep='\t')

union_bc_h5_rna_ = h5_rna_[h5_rna_.obs['atac_barcodes'].isin(union_bc_df['atac_bc'])].copy()

union_bc_h5_rna_.write_h5ad(os.path.join(entropy_dir, '_cell_calling_comparison/Union_3set_rna.h5ad'))

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/hdd_bob/syy/adipose/atac/res/VIB_10xmultiome_2_WS3000F/_cell_calling_comparison/Union_cell_3set.tsv'

In [14]:
union_bc_h5_rna_

NameError: name 'union_bc_h5_rna_' is not defined